# Module 6 Homework

In this homework we'll put what we learned about Spark in practice.

For this homework we will be using the Yellow 2025-11 data from the official website:

```bash
wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
```

## Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

```bash
sudo apt update
sudo apt install openjdk-21-jdk
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))
export PATH="${JAVA_HOME}/bin:${PATH}"
```

What's the output?

> [!NOTE]
> To install PySpark follow this [guide](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/06-batch/setup/pyspark.md)

In [9]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = "/usr/lib/jvm/java-17-openjdk-amd64/bin:" + os.environ["PATH"]
import pyspark
pyspark.__version__

'4.1.1'

## Question 2: Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()
    
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df.printSchema()


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [11]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [12]:
df = df.repartition(4)
df.write.parquet('yellow/2025/11/', mode='overwrite')

In [13]:
!ls -lh /workspaces/DataTalksClub-Zoomcamp/data-engineering-zoomcamp/2026/Homework6/yellow/2025/11/*.parquet

-rw-r--r-- 1 codespace codespace 25M Feb 22 06:46 /workspaces/DataTalksClub-Zoomcamp/data-engineering-zoomcamp/2026/Homework6/yellow/2025/11/part-00000-020c22c2-2331-4ec0-a3fc-01597c4abd59-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 25M Feb 22 06:46 /workspaces/DataTalksClub-Zoomcamp/data-engineering-zoomcamp/2026/Homework6/yellow/2025/11/part-00001-020c22c2-2331-4ec0-a3fc-01597c4abd59-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 25M Feb 22 06:46 /workspaces/DataTalksClub-Zoomcamp/data-engineering-zoomcamp/2026/Homework6/yellow/2025/11/part-00002-020c22c2-2331-4ec0-a3fc-01597c4abd59-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 25M Feb 22 06:46 /workspaces/DataTalksClub-Zoomcamp/data-engineering-zoomcamp/2026/Homework6/yellow/2025/11/part-00003-020c22c2-2331-4ec0-a3fc-01597c4abd59-c000.snappy.parquet


## Question 3: Count records

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

In [14]:
df.createOrReplaceTempView("yellow_taxi")

spark.sql("""
    SELECT COUNT(*) AS trip_count
    FROM yellow_taxi
    WHERE DATE(tpep_pickup_datetime) = '2025-11-15'
""").show()

+----------+
|trip_count|
+----------+
|    162604|
+----------+



## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

In [15]:
spark.sql("""
    SELECT MAX((UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 3600) AS longest_trip_hours
    FROM yellow_taxi
""").show()

+------------------+
|longest_trip_hours|
+------------------+
| 90.64666666666666|
+------------------+



## Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

Answer: 4040

## Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

In [18]:
zones = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)
zones.createOrReplaceTempView("zones")

spark.sql("""
    SELECT z.Zone, COUNT(t.PULocationID) AS pickup_count
    FROM zones z
    LEFT JOIN yellow_taxi t ON z.LocationID = t.PULocationID
    GROUP BY z.Zone
    ORDER BY pickup_count ASC
    LIMIT 10
""").show(truncate=False)

+---------------------------------------------+------------+
|Zone                                         |pickup_count|
+---------------------------------------------+------------+
|Freshkills Park                              |0           |
|Charleston/Tottenville                       |0           |
|Great Kills Park                             |0           |
|Governor's Island/Ellis Island/Liberty Island|1           |
|Eltingville/Annadale/Prince's Bay            |1           |
|Arden Heights                                |1           |
|Port Richmond                                |3           |
|Rossville/Woodrow                            |4           |
|Rikers Island                                |4           |
|Great Kills                                  |4           |
+---------------------------------------------+------------+



## Submitting the solutions

- Form for submitting: https://courses.datatalks.club/de-zoomcamp-2026/homework/hw6